In [1]:
!pip install tqdm

import nltk as nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from tqdm import tqdm
from collections import defaultdict, Counter
import numpy as np
import math as math

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

stop_words = set(stopwords.words('english'))
df = pd.DataFrame(pd.read_json('/content/drive/MyDrive/Information_Retrieval/BM25/data/corpus.jsonl', lines=True))
df.drop(columns=['metadata'], inplace=True)
corpus_tokens = {}

def tokenize(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return filtered_tokens

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    tokens = tokenize(row['text'])
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    corpus_tokens[row['_id']] = filtered_tokens

100%|██████████| 171332/171332 [03:36<00:00, 791.01it/s]


In [4]:
inverted_index = defaultdict(dict)
for doc_id, tokens in tqdm(corpus_tokens.items(), desc='Indexing...'):
    for term, frequency in Counter(tokens).items():
        inverted_index[term][doc_id] = frequency

Indexing...: 100%|██████████| 171332/171332 [00:16<00:00, 10313.47it/s]


In [5]:
docs_len = {}
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc='Calculating doc stats...'):
    docs_len[row['_id']] = len(corpus_tokens[row['_id']])

Calculating doc stats...: 100%|██████████| 171332/171332 [00:09<00:00, 18048.81it/s]


In [6]:
N = len(df)
average_dl = sum(docs_len.values()) / N

def bm25_score(term, doc_id, k1=0.5, b=1.0):
  if term not in inverted_index or doc_id not in inverted_index[term]:
    return 0.0

  tf = inverted_index[term][doc_id]
  dl = docs_len[doc_id]
  df = len(inverted_index[term])
  idf = math.log((N - df + 0.5) / (df + 0.5))
  denom = tf + k1 * (1 - b + b * dl / average_dl)
  score = idf * (tf * (k1 + 1) / denom)
  return score


In [7]:
query = 'what is the origin of COVID-19'
query_tokens = tokenize(query)
union_docs = set().union(*(inverted_index[t].keys() for t in query_tokens))

scores = defaultdict(float)
for doc_id in tqdm(union_docs, desc='Calculating scores...'):
    score = sum(bm25_score(t, doc_id) for t in query_tokens)
    scores[doc_id] = score

sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
sorted_scores = sorted_scores[:50]
sorted_scores

Calculating scores...: 100%|██████████| 2048/2048 [00:00<00:00, 415635.29it/s]


[('2vpvdm11', 6.40731286223402),
 ('021q9884', 6.40731286223402),
 ('dv9m19yk', 6.313619794156226),
 ('vh96sjss', 6.199882886270728),
 ('e3wjo0yk', 6.110543742557887),
 ('7csfkoh8', 6.079257686372115),
 ('h68eivx4', 5.957252866027686),
 ('n15i01tn', 5.957252866027686),
 ('47pszpgp', 5.957252866027686),
 ('ezi2mret', 5.927513003187837),
 ('wqcexgnh', 5.927513003187837),
 ('23cdi61w', 5.898068600684016),
 ('jn3xni1u', 5.898068600684016),
 ('pxwopt88', 5.8689152772595845),
 ('4uaa6kpg', 5.847238718409724),
 ('qyf59ghf', 5.840048737855584),
 ('l8zkeyxi', 5.840048737855584),
 ('8ccl9aui', 5.82858147558405),
 ('gy8d8285', 5.811464771501271),
 ('l0kc731z', 5.811464771501271),
 ('deajwhx0', 5.783159249266305),
 ('icwvm7jp', 5.783159249266305),
 ('cniyembt', 5.783159249266305),
 ('hpcni41t', 5.783159249266305),
 ('zd7smm8r', 5.769109636345279),
 ('dg5pc3a0', 5.755128122272484),
 ('d0x23frk', 5.748162747767869),
 ('ayj4z8qn', 5.7412142131295365),
 ('v6ci69n0', 5.736591182051774),
 ('l1efivdt', 5